# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

features = pd.DataFrame()
features['impressions_90d'] = df['impressions_90d']
features['clicks_90d'] = df['clicks_90d']
features['avg_position'] = df['avg_position']
features['word_count'] = df['word_count']
features['days_since_last_update'] = df['days_since_last_update']

features = pd.concat([features, pd.get_dummies(df['content_type'], prefix='type')], axis=1)
features = features.fillna(0)

features.head()

,impressions_90d,clicks_90d,avg_position,word_count,days_since_last_update,type_comparison article,type_feedly article,type_keyword article
0,3803,29,10.6,3221.0,20,False,False,True
1,15320,7,20.3,2481.0,25,False,False,True
2,12581,11,36.5,3515.0,20,False,False,True
3,11751,58,6.2,0.0,22,False,False,True
4,19140,24,44.0,2803.0,14,False,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_90d — how often the page appeared in search over 90 days. Missing values filled with 0 (means no visibility). Known before prediction, since it only uses past data.
clicks_90d — how many times the page was clicked. Filled with 0 if missing. Known before prediction.
avg_position — average search ranking position. Filled with 0 if missing (treated as "no ranking data"). Known before prediction.
word_count — length of the page content. Filled with 0 if missing. Known before prediction, since content doesn't change based on future outcomes.
days_since_last_update — how long since the page was last edited. Filled with 0 if missing. Known before prediction.
content_type (one-hot) — category of the page (blog, product, etc). No missing values expected. Known before prediction, since it's fixed content metadata.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y = (df['trend_direction'] == 'down').astype(int)

leaky = features.copy()
leaky['leaky_column'] = y

X_train, X_test, y_train, y_test = train_test_split(leaky, y, test_size=0.2, random_state=42)
model_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_leak.fit(X_train, y_train)
print("AUC with leaky column:", roc_auc_score(y_test, model_leak.predict_proba(X_test)[:, 1]))

clean = leaky.drop(columns=['leaky_column'])

X_train, X_test, y_train, y_test = train_test_split(clean, y, test_size=0.2, random_state=42)
model_clean = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_clean.fit(X_train, y_train)
print("AUC after removing it:", roc_auc_score(y_test, model_clean.predict_proba(X_test)[:, 1]))

AUC with leaky column: 1.0
AUC after removing it: 0.7228008274030442


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- health_score, priority_score, action_type: these are FlyRank's own product decisions, not raw signals. Using them would let my model copy the existing rule instead of learning from data.
- trend_direction and trend_pct: trend_direction is my label, and trend_pct is calculated from the same trend, so both would give away the answer.
- content_id and client_id: these are only identifiers for joining and grouping, and they carry no real signal about decline.
- impressions_last_30d, clicks_last_30d and the prev_30d columns: they are built from the same window the label is calculated from, so they could leak the answer.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.